# dsRAG Integration Sandbox

**Objective:** Test dsRAG for contextual chunking and retrieval on multi-page documents.

**What we'll test:**
1. Create a KnowledgeBase with OpenAI embeddings and Cohere reranker
2. Add a multi-page document with semantic sectioning
3. Query the KB and inspect retrieved segments
4. Connect retrieval to OpenAI structured extraction

**Prerequisites:**
- `OPENAI_API_KEY` and `CO_API_KEY` environment variables set
- dsRAG installed (`pip install dsrag`)

In [2]:
!pip install python-dotenv

  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
Using cached python_dotenv-1.2.1-py3-none-any.whl (21 kB)


In [2]:
# Cell 1: Setup - Load environment and imports
import os
import sys
from pathlib import Path
from pprint import pprint

# Add backend to path for accessing project utilities
backend_path = Path.cwd().parent.parent / "backend" / "src"
if str(backend_path) not in sys.path:
    sys.path.insert(0, str(backend_path))

# Load environment variables
from dotenv import load_dotenv
load_dotenv(Path.cwd().parent.parent / ".env")

# Verify required API keys are set
required_keys = ["OPENAI_API_KEY", "CO_API_KEY"]
for key in required_keys:
    if os.getenv(key):
        print(f"✓ {key} is set")
    else:
        print(f"✗ {key} is NOT set - please configure in .env")

# dsRAG imports
from dsrag.knowledge_base import KnowledgeBase
from dsrag.llm import OpenAIChatAPI
from dsrag.reranker import CohereReranker, NoReranker
from dsrag.embedding import OpenAIEmbedding

print("\ndsRAG imports successful!")

✓ OPENAI_API_KEY is set
✓ CO_API_KEY is set


c:\Projects\test\data-labeller\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Projects\test\data-labeller\backend\.venv\Lib\site-packages\dsrag\llm.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai



dsRAG imports successful!


## Configuration

Set up paths and dsRAG storage location.

In [3]:
# Cell 2: Configuration
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data"
DSRAG_STORAGE_DIR = DATA_DIR / "dsrag"
FILES_DIR = DATA_DIR / "files"

# Create dsRAG storage directory if it doesn't exist
DSRAG_STORAGE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"dsRAG storage: {DSRAG_STORAGE_DIR}")
print(f"Files directory: {FILES_DIR}")

# List available PDF files for testing
pdf_files = list(FILES_DIR.glob("**/*.pdf")) if FILES_DIR.exists() else []
print(f"\nAvailable PDF files ({len(pdf_files)}):")
for pdf in pdf_files[:10]:
    print(f"  - {pdf.relative_to(FILES_DIR)}")

Project root: c:\Projects\test\data-labeller
dsRAG storage: c:\Projects\test\data-labeller\data\dsrag
Files directory: c:\Projects\test\data-labeller\data\files

Available PDF files (0):


## Create KnowledgeBase

Initialize dsRAG KnowledgeBase with:
- **OpenAI Embeddings** (text-embedding-3-small)
- **Cohere Reranker** (rerank-v3.5)
- **OpenAI LLM** for AutoContext (gpt-4o-mini)

In [5]:
# Cell 3: Create KnowledgeBase with OpenAI + Cohere
KB_ID = "data_labeller_sandbox"

# Initialize components
llm = OpenAIChatAPI(model="gpt-4o-mini")
reranker = CohereReranker()  # Uses CO_API_KEY from environment

# Create or load existing KnowledgeBase
# KnowledgeBases persist automatically - if KB_ID exists, it will be loaded
kb = KnowledgeBase(
    kb_id=KB_ID,
    storage_directory=str(DSRAG_STORAGE_DIR),
    reranker=reranker,
    auto_context_model=llm,
)

print(f"KnowledgeBase '{KB_ID}' initialized")
print(f"Storage directory: {kb.storage_directory}")

KnowledgeBase 'data_labeller_sandbox' initialized
Storage directory: c:\Projects\test\data-labeller\data\dsrag


In [6]:
# Cell 3b (Optional): Azure Cohere Reranker
# Use this if the standard Cohere reranker doesn't work with your Azure endpoint

import requests
from dsrag.reranker import Reranker

class AzureCohereReranker(Reranker):
    """Custom reranker for Azure-hosted Cohere API."""
    
    def __init__(
        self,
        api_key: str = None,
        endpoint: str = None,
        model: str = "Cohere-rerank-v3.5-turbo-128k",
    ):
        self.api_key = api_key or os.getenv("CO_API_KEY")
        self.endpoint = endpoint or os.getenv(
            "CO_RERANK_ENDPOINT",
            "https://mbaistudio3062596349.services.ai.azure.com/providers/cohere/v2/rerank"
        )
        self.model = model
    
    def rerank(
        self,
        query: str,
        documents: list[str],
        top_n: int = 10,
    ) -> list[dict]:
        """Rerank documents using Azure Cohere endpoint."""
        if not documents:
            return []
        
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }
        
        payload = {
            "model": self.model,
            "query": query,
            "documents": documents,
            "top_n": min(top_n, len(documents)),
        }
        
        response = requests.post(self.endpoint, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        
        result = response.json()
        
        # Convert to dsRAG expected format
        reranked = []
        for item in result.get("results", []):
            reranked.append({
                "index": item["index"],
                "relevance_score": item["relevance_score"],
            })
        
        return reranked
    
    def to_dict(self) -> dict:
        return {
            "type": "AzureCohereReranker",
            "model": self.model,
            "endpoint": self.endpoint,
        }

# Uncomment below to use Azure Cohere instead of standard Cohere:
# reranker = AzureCohereReranker()
# kb = KnowledgeBase(
#     kb_id=KB_ID,
#     storage_directory=str(DSRAG_STORAGE_DIR),
#     reranker=reranker,
#     auto_context_model=llm,
# )

print("AzureCohereReranker class defined (use if standard Cohere doesn't work)")

AzureCohereReranker class defined (use if standard Cohere doesn't work)


## Add Document to KnowledgeBase

Add a multi-page PDF with semantic sectioning and chunking.

**Key configurations:**
- `semantic_sectioning_config`: Groups content by document structure
- `chunking_config`: Controls chunk size and overlap
- `auto_context_config`: Enables document/section summaries for better context

In [9]:
# Cell 4: Add a test document
# Set the path to your test PDF here
# TEST_PDF_PATH = FILES_DIR / "your-document.pdf"

# For testing, you can also provide a URL or use a sample document
TEST_PDF_PATH = 'C:/Projects/10k/intc.pdf'  # Set this to your PDF path

if TEST_PDF_PATH is None and pdf_files:
    TEST_PDF_PATH = pdf_files[0]
    print(f"Using first available PDF: {TEST_PDF_PATH}")
elif TEST_PDF_PATH is None:
    print("No PDF files found. Please set TEST_PDF_PATH manually.")
    print("Example: TEST_PDF_PATH = Path('/path/to/your/document.pdf')")

if TEST_PDF_PATH and Path(TEST_PDF_PATH).exists():
    print(f"\nAdding document: {TEST_PDF_PATH}")
    print("This may take a few minutes for large documents...")
else:
    print("Skipping document addition - no valid PDF path")


Adding document: C:/Projects/10k/intc.pdf
This may take a few minutes for large documents...


In [10]:
# Cell 5: Index document with semantic sectioning
if TEST_PDF_PATH and Path(TEST_PDF_PATH).exists():
    doc_id = Path(TEST_PDF_PATH).stem  # Use filename without extension as doc_id
    
    # Configuration for semantic sectioning
    semantic_sectioning_config = {
        "use_semantic_sectioning": True,
        "llm_provider": "openai",
        "model": "gpt-4o-mini",
    }
    
    # Configuration for chunking
    chunking_config = {
        "chunk_size": 1000,  # Max characters per chunk
        "min_length_for_chunking": 500,  # Don't chunk very short sections
    }
    
    # AutoContext configuration - generates summaries for better retrieval
    auto_context_config = {
        "use_generated_title": True,
        "get_document_summary": True,
        "get_section_summaries": True,  # Important for long documents
    }
    
    print(f"Indexing document: {doc_id}")
    print(f"  Semantic sectioning: {semantic_sectioning_config['use_semantic_sectioning']}")
    print(f"  Chunk size: {chunking_config['chunk_size']}")
    print(f"  Section summaries: {auto_context_config['get_section_summaries']}")
    
    # Add document to KnowledgeBase
    kb.add_document(
        doc_id=doc_id,
        file_path=str(TEST_PDF_PATH),
        semantic_sectioning_config=semantic_sectioning_config,
        chunking_config=chunking_config,
        auto_context_config=auto_context_config,
    )
    
    print(f"\n✓ Document '{doc_id}' indexed successfully!")
else:
    print("No document to index. Set TEST_PDF_PATH in the previous cell.")

Indexing document: intc
  Semantic sectioning: True
  Chunk size: 1000
  Section summaries: True
File not found: c:\Projects\test\data-labeller\data\dsrag\page_images\data_labeller_sandbox\intc\elements.json

✓ Document 'intc' indexed successfully!


In [11]:
# Cell 6: Inspect indexed chunks
if TEST_PDF_PATH and Path(TEST_PDF_PATH).exists():
    doc_id = Path(TEST_PDF_PATH).stem
    
    # Get chunk database info
    chunk_db = kb.chunk_db
    
    # List all chunks for the document
    if hasattr(chunk_db, 'get_all_chunks'):
        chunks = chunk_db.get_all_chunks(doc_id)
    else:
        # For BasicChunkDB, access the internal structure
        chunks = chunk_db.data.get(doc_id, {})
    
    print(f"Document: {doc_id}")
    print(f"Total chunks: {len(chunks)}")
    
    # Show first few chunks
    print("\n--- Sample Chunks ---")
    chunk_items = list(chunks.items()) if isinstance(chunks, dict) else list(enumerate(chunks))
    for idx, (chunk_idx, chunk_data) in enumerate(chunk_items[:3]):
        if isinstance(chunk_data, dict):
            text = chunk_data.get('text', chunk_data.get('content', str(chunk_data)))[:200]
        else:
            text = str(chunk_data)[:200]
        print(f"\nChunk {chunk_idx}:")
        print(f"  {text}...")
else:
    print("No document indexed yet.")

Document: intc
Total chunks: 1036

--- Sample Chunks ---

Chunk 0:
  {'chunk_text': "UNITED STATES SECURITIES AND EXCHANGE COMMISSIONWashington, D.C. 20549\nFORM 10-K(Mark One)\n☑ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934For t...

Chunk 1:
  {'chunk_text': 'Common stock, $0.001 par value INTC Nasdaq Global Select Market\nSecurities registered pursuant to Section 12(g) of the Act: None\nIndicate by check mark if the registrant is a well-kn...

Chunk 2:
  {'chunk_text': 'duringthe preceding 12 months (or for such shorter period that the registrant was required to submit such files). Yes ☑ No ☐\nIndicate by check mark whether the registrant is a large a...


In [24]:
print(chunks[7]['chunk_text'])

Forward-Looking Statements
This Form 10-K contains forward-looking statements that involve a number of risks and uncertainties. Words such as "accelerate", "achieve", "aim", "ambitions", "anticipate","believe", "committed",
"continue", "could", "designed", "estimate", "expect", "forecast", "future", "goals", "grow", "guidance", "intend", "likely", "may", "might", "milestones", "nextgeneration", "objective", "on track",
"opportunity", "outlook", "pending", "plan", "position", "possible", "potential", "predict", "progress", "ramp", "roadmap", "seek", "should", "strive","targets", "to be", "upcoming", "will", "would",
and variations of such words and similar expressions are intended to identify such forward-looking statements, which may includestatements regarding:
▪ our business plans, strategy and leadership and anticipated benefits therefrom, including with respect to our foundry strategy, Smart Capital strategy, partnerships withApollo and Brookfield, AI


## Query the KnowledgeBase

Test retrieval with schema-related queries. dsRAG uses RSE (Relevant Segment Extraction) 
to find the most relevant contiguous segments of text.

In [25]:
# Cell 7: Query the KnowledgeBase
# Define test queries - customize these for your document schema

TEST_QUERIES = [
    "What were intel's strategy"
]

# RSE parameters for retrieval
rse_params = {
    "max_length": 10,  # Max chunks per segment
    "overall_max_length": 20,  # Max total chunks across all segments
    "minimum_value": 0.3,  # Minimum relevance score
}

print("Running queries against KnowledgeBase...")
print("=" * 60)

query_results = {}

for query in TEST_QUERIES:
    print(f"\nQuery: {query}")
    print("-" * 40)
    
    results = kb.query(
        search_queries=[query],
        rse_params=rse_params,
    )
    
    query_results[query] = results
    
    if results:
        print(f"Found {len(results)} relevant segment(s)")
        for i, segment in enumerate(results[:2]):  # Show top 2 segments
            # Truncate long text for display
            text = segment.get('text', str(segment))[:300]
            print(f"\n  Segment {i+1}:")
            print(f"    {text}...")
    else:
        print("  No relevant segments found")

print("\n" + "=" * 60)
print("Query testing complete!")

Running queries against KnowledgeBase...

Query: What were intel's strategy
----------------------------------------


UnauthorizedError: headers: {'cache-control': 'no-cache, no-store, no-transform, must-revalidate, private, max-age=0', 'content-encoding': 'gzip', 'content-type': 'application/json', 'expires': 'Thu, 01 Jan 1970 00:00:00 GMT', 'pragma': 'no-cache', 'vary': 'Origin,Accept-Encoding', 'x-accel-expires': '0', 'date': 'Thu, 19 Feb 2026 15:59:45 GMT', 'server': 'envoy', 'via': '1.1 google', 'alt-svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000', 'connection': 'close', 'transfer-encoding': 'chunked'}, status_code: 401, body: {'id': '65c37bef-d0d4-4a73-9813-b1e0d4cf251d', 'message': 'invalid api token'}

## Integration: Retrieval + Structured Extraction

Combine dsRAG retrieval with OpenAI structured output to extract schema fields
from only the most relevant document segments.

In [ ]:
# Cell 8: Define extraction schema using Pydantic
from pydantic import BaseModel, Field
from typing import Optional, List
from openai import OpenAI

# Example extraction schema - customize for your use case
class LineItem(BaseModel):
    description: str = Field(description="Description of the item")
    quantity: Optional[float] = Field(None, description="Quantity of the item")
    unit_price: Optional[float] = Field(None, description="Price per unit")
    total: Optional[float] = Field(None, description="Total price for this line")

class InvoiceExtraction(BaseModel):
    """Schema for extracting invoice data."""
    invoice_number: Optional[str] = Field(None, description="Invoice number or ID")
    invoice_date: Optional[str] = Field(None, description="Date of the invoice")
    vendor_name: Optional[str] = Field(None, description="Name of the vendor/seller")
    customer_name: Optional[str] = Field(None, description="Name of the customer/buyer")
    subtotal: Optional[float] = Field(None, description="Subtotal before tax")
    tax_amount: Optional[float] = Field(None, description="Tax amount")
    total_amount: Optional[float] = Field(None, description="Total amount due")
    line_items: List[LineItem] = Field(default_factory=list, description="List of line items")

print("Extraction schema defined: InvoiceExtraction")
print(f"Fields: {list(InvoiceExtraction.model_fields.keys())}")

In [ ]:
# Cell 9: RAG-Enhanced Extraction Function
def extract_with_rag(
    kb: KnowledgeBase,
    schema: type[BaseModel],
    extraction_queries: list[str],
    model: str = "gpt-4o-mini",
) -> BaseModel:
    """
    Use dsRAG to find relevant segments, then extract structured data.
    
    Args:
        kb: The dsRAG KnowledgeBase
        schema: Pydantic model defining the extraction schema
        extraction_queries: Queries to find relevant content
        model: OpenAI model to use for extraction
    
    Returns:
        Extracted data as a Pydantic model instance
    """
    # Step 1: Retrieve relevant segments using dsRAG
    print("Step 1: Retrieving relevant segments...")
    segments = kb.query(
        search_queries=extraction_queries,
        rse_params={
            "max_length": 15,
            "overall_max_length": 30,
            "minimum_value": 0.2,
        },
    )
    
    if not segments:
        print("  No relevant segments found!")
        return schema()
    
    print(f"  Found {len(segments)} relevant segment(s)")
    
    # Step 2: Combine segments into context
    context_parts = []
    for i, segment in enumerate(segments):
        text = segment.get('text', str(segment))
        context_parts.append(f"[Segment {i+1}]\n{text}")
    
    context = "\n\n".join(context_parts)
    total_chars = len(context)
    print(f"  Total context: {total_chars:,} characters")
    
    # Step 3: Extract using OpenAI structured output
    print("Step 2: Extracting structured data...")
    client = OpenAI()
    
    completion = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You are a document extraction assistant. Extract the requested information from the provided document segments. Only include information that is explicitly stated in the text."
            },
            {
                "role": "user",
                "content": f"Extract the following information from these document segments:\n\n{context}"
            }
        ],
        response_format=schema,
    )
    
    result = completion.choices[0].message.parsed
    print("  Extraction complete!")
    
    return result

print("extract_with_rag() function defined")

In [ ]:
# Cell 10: Run RAG-enhanced extraction
# Define queries that will help find relevant content for our schema
EXTRACTION_QUERIES = [
    "Find the invoice number, date, and total amount",
    "What are the vendor and customer names and addresses?",
    "List all line items with quantities and prices",
    "What is the subtotal, tax, and total?",
]

print("Running RAG-enhanced extraction...")
print("=" * 60)

extraction_result = extract_with_rag(
    kb=kb,
    schema=InvoiceExtraction,
    extraction_queries=EXTRACTION_QUERIES,
)

print("\n" + "=" * 60)
print("EXTRACTION RESULTS:")
print("=" * 60)
print(extraction_result.model_dump_json(indent=2))

## Utilities and Analysis

Helper functions for managing the KnowledgeBase and analyzing results.

In [ ]:
# Cell 11: Utility functions

def list_indexed_documents(kb: KnowledgeBase) -> list[str]:
    """List all documents in the KnowledgeBase."""
    if hasattr(kb.chunk_db, 'data'):
        return list(kb.chunk_db.data.keys())
    return []

def delete_document(kb: KnowledgeBase, doc_id: str):
    """Remove a document from the KnowledgeBase."""
    kb.delete_document(doc_id)
    print(f"Deleted document: {doc_id}")

def clear_knowledge_base(kb: KnowledgeBase):
    """Clear all documents from the KnowledgeBase."""
    docs = list_indexed_documents(kb)
    for doc_id in docs:
        kb.delete_document(doc_id)
    print(f"Cleared {len(docs)} document(s) from KnowledgeBase")

# Show indexed documents
docs = list_indexed_documents(kb)
print(f"Currently indexed documents ({len(docs)}):")
for doc in docs:
    print(f"  - {doc}")

## Summary and Next Steps

### What we tested:
1. **KnowledgeBase creation** with OpenAI embeddings and Cohere reranker
2. **Document indexing** with semantic sectioning and AutoContext
3. **Query retrieval** using RSE (Relevant Segment Extraction)
4. **RAG-enhanced extraction** combining retrieval with structured output

### Benefits for large documents:
- **Reduced token usage**: Only process relevant segments, not entire document
- **Better accuracy**: Focus LLM attention on most relevant content
- **Scalable**: Works with 200+ page documents without hitting context limits

### Next steps for production integration:
1. Create `DsRAGService` in `backend/src/uu_backend/services/`
2. Add Celery task for async document indexing
3. Integrate with `ExtractionService` for schema-based extraction
4. Configure ChromaDB or other vector DB for persistence

### Configuration tuning:
- Adjust `chunk_size` based on your document structure (1000-1500 chars)
- Enable `get_section_summaries` for long documents
- Tune `rse_params` based on extraction quality needs